In [ ]:
import math
import numpy as np

# Hidden states: s = start, E = exon, 5 = splice-site, I = intron, e = end
STATE_NAMES = ["s", "E", "5", "I", "e"]
STATE_TO_ID  = {name: idx for idx, name in enumerate(STATE_NAMES)}
ID_TO_STATE  = {idx: name for idx, name in enumerate(STATE_NAMES)}

# encodings
NUC_TO_ID = {"A": 0, "C": 1, "G": 2, "T": 3}

# Transition matrix
TRANSITION = np.array([
    [0.0, 1.0, 0.0, 0.0, 0.0],   # s  → ...
    [0.0, 0.9, 0.1, 0.0, 0.0],   # E  → ...
    [0.0, 0.0, 0.0, 1.0, 0.0],   # 5  → ...
    [0.0, 0.0, 0.0, 0.9, 0.1],   # I  → ...
    [0.0, 0.0, 0.0, 0.0, 0.0],   # e  → ...
])

# Emission matrix  (columns = A, C, G, T)
EMISSION = np.array([
    [0.00, 0.00, 0.00, 0.00],   # s  — silent state
    [0.25, 0.25, 0.25, 0.25],   # E  — uniform
    [0.05, 0.00, 0.95, 0.00],   # 5  — GU splice donor
    [0.40, 0.10, 0.10, 0.40],   # I  — AT-rich intron
    [0.00, 0.00, 0.00, 0.00],   # e  — silent state
])

# Observed sequence
QUERY = "CTTCATGTGAAAGCAGACGTAAGTCA"

#  Log-probability of an explicit state path 

def log_prob_of_path(state_path: str, sequence: str) -> float:
    """
    Return the log-probability of a given state path producing the sequence.
    The first position is initialised with log(0.25) (uniform start over nucleotides).
    """
    log_p = math.log(0.25)
    for i in range(1, len(state_path)):
        prev_id = STATE_TO_ID[state_path[i - 1]]
        curr_id = STATE_TO_ID[state_path[i]]
        nuc_id  = NUC_TO_ID[sequence[i]]

        trans = TRANSITION[prev_id][curr_id]
        emit  = EMISSION[curr_id][nuc_id]
        log_p += math.log(trans * emit)

    return log_p


# Viterbi: initialisation

def initialise_viterbi(sequence: str):
    """
    Set up the Viterbi value matrix (log-probs) and traceback matrix.
    Both have shape (num_states × sequence_length).
    """
    n_states = len(STATE_NAMES)
    seq_len  = len(sequence)

    values  = np.full((n_states, seq_len), -math.inf)
    pointers = np.full((n_states, seq_len), -1, dtype=int)

    first_nuc_id = NUC_TO_ID[sequence[0]]

    for state_id in range(n_states):
        emit = EMISSION[state_id][first_nuc_id]
        if emit > 0:
            values[state_id][0] = math.log(0.25) + math.log(emit)
        pointers[state_id][0] = state_id   # points to itself at t=0

    return values, pointers


# Viterbi: fill
def fill_viterbi(sequence: str, values: np.ndarray, pointers: np.ndarray):
    """
    Fill the Viterbi matrices column by column (left → right).
    For each (current_state, position) cell we find the predecessor state
    that maximises:  values[prev][t-1] + log P(prev→curr) + log P(curr emits nuc)
    """
    for t in range(1, len(sequence)):
        nuc_id = NUC_TO_ID[sequence[t]]

        for curr_id in range(len(STATE_NAMES)):
            emit = EMISSION[curr_id][nuc_id]
            if emit == 0:
                continue                          # impossible emission — skip

            best_score = -math.inf
            best_prev  = -1

            for prev_id in range(len(STATE_NAMES)):
                trans      = TRANSITION[prev_id][curr_id]
                prev_score = values[prev_id][t - 1]

                if trans == 0 or prev_score == -math.inf:
                    continue

                score = prev_score + math.log(trans) + math.log(emit)
                if score > best_score:
                    best_score = score
                    best_prev  = prev_id

            values[curr_id][t]   = best_score
            pointers[curr_id][t] = best_prev


#  Viterbi: traceback 

def traceback(values: np.ndarray, pointers: np.ndarray) -> list[str]:
    """
    Walk backwards through the pointer matrix to recover the best state path.
    """
    _, seq_len = values.shape

    # Start from the state with the highest log-prob at the last position
    current = int(np.argmax(values[:, seq_len - 1]))
    path    = [current]

    for t in range(seq_len - 1, 0, -1):
        current = pointers[current][t]
        path.append(current)

    path.reverse()
    return [ID_TO_STATE[sid] for sid in path]


#  Main 

if __name__ == "__main__":
    # Compare a few hand-crafted paths
    candidate_paths = {
        "EEEEEE5IIIIIIIIIIIIIIIIIII": "5-site at position  6",
        "EEEEEEEE5IIIIIIIIIIIIIIIII": "5-site at position  8",
        "EEEEEEEEEEEE5IIIIIIIIIIIII": "5-site at position 12",
        "EEEEEEEEEEEEEEE5IIIIIIIIII": "5-site at position 15",
        "EEEEEEEEEEEEEEEEEE5IIIIIII": "5-site at position 18",
        "EEEEEEEEEEEEEEEEEEEEEE5III": "5-site at position 22",
        "EEEEEEEEEEEEEEEEEEEEEEEEEE": "all-exon",
    }

    print("Log-probabilities for candidate paths:")
    for path, label in candidate_paths.items():
        lp = log_prob_of_path(path, QUERY) + math.log(0.1)
        print(f"  {label:<24}  {lp:.4f}")

    print()

    # Run Viterbi
    val_matrix, ptr_matrix = initialise_viterbi(QUERY)
    fill_viterbi(QUERY, val_matrix, ptr_matrix)
    best_path = traceback(val_matrix, ptr_matrix)

    print("Query sequence :  ", QUERY)
    print("Best state path:  ", "".join(best_path))

Log-probabilities for candidate paths:
  5-site at position  6     -43.8974
  5-site at position  8     -43.4511
  5-site at position 12     -43.9448
  5-site at position 15     -42.5823
  5-site at position 18     -41.2197
  5-site at position 22     -41.7134
  all-exon                  -40.9803

Query sequence :   CTTCATGTGAAAGCAGACGTAAGTCA
Best state path:   EEEEEEEEEEEEEEEEEEEEEEEEEE
